<a href="https://colab.research.google.com/github/J0927/NLP_2026_spring/blob/main/nlp_as4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 作業四：LLM Function Calling

本作業要實作一個 **Function Calling AI Agent**，核心是下面的兩輪流程：

```
使用者提問
  → Round 1：LLM 判斷要不要呼叫工具，輸出 JSON
      • 需要工具：{"tool": "...", "arguments": {...}}
      • 不需要工具（閒聊）：{"final": "..."}
  → Python 路由：依工具名查 mock_data.json
  → Round 2：把查到的資料丟回 LLM，要它輸出自然語言 {"final": "..."}
```

你需要練習的不是寫聊天機器人，而是：

1. 用 **system prompt** 約束模型「該用工具就用、不該用就閒聊」並只吐 JSON。
2. 設計 **tool schema**，讓模型在工具中選對的、抽對的參數。
3. 在 Python 中寫 **handle_tool_call**，把模型輸出路由到對應的查詢函式。
4. 設計一個你自己的 **custom tool**，從現有資料推導新結果。

## 環境準備

1. 到 [console.groq.com](https://console.groq.com) 免費申請 API Key
2. `cp .env.example .env`，把 key 貼進 `.env` 的 `GROQ_API_KEY=`

## 評分
- 共 110 分
- Q1–Q5 共 50 分（每題 10 分）
- 自訂工具 10 分
- 報告 40 分
- 加分題：使用兩個以上模型比較 +10

## 0. 載入工具與資料庫

`fc_utils.py` 是助教提供的共用工具（你不需要修改它），裡面有：

- `call_llm(messages)`：呼叫 Groq Qwen3-32B 或其他模型
- `parse_json_output(text)`：從模型輸出抓 JSON
- `lookup_db(section, key)`：查 mock_data 的工具函式
- `load_db(path)`：載入 mock_data.json
- `run_tests(run_agent, db)`：跑 5 題並輸出 `fc_log.json`

In [ ]:
!pip install groq

In [ ]:
import json
from fc_utils import (
    call_llm,
    parse_json_output,
    lookup_db,
    load_db,
    run_tests,
    set_model,
)

db = load_db("mock_data.json")
print("資料庫類別：", list(db.keys()))
print("範例（AAPL）：", db["stocks"]["data"]["AAPL"])

In [ ]:
# TODO1: 指定模型名稱
# 加分題（+10）：在報告中比較至少兩個模型的結果。
# 請分別跑 run_tests，把產出檔重新命名為 fc_log_<模型名>.json 一併繳交，並在報告中分析差異。

set_model("")  # https://console.groq.com/docs/models

## 1. Round 1 System Prompt

Round 1 的目標：讓模型看到使用者問題後，**只**輸出一個 JSON。可能是：

- 需要呼叫工具：`{"tool": "...", "arguments": {...}}`
- 閒聊或與工具無關：`{"final": "<繁體中文回答>"}`

好的 prompt 通常會：

1. 列出所有可用工具與其參數格式
2. 用範例示範兩種 JSON 格式
3. 強制不准輸出 JSON 以外的東西
4. 強制不准猜測股價、天氣（必須靠工具）

In [ ]:
# TODO2: 請修改 SYSTEM_PROMPT_R1，目前已經提供最基礎的 prompt
# 請完善它使模型能順利進行 function calling

SYSTEM_PROMPT_R1 = """你是 Function Calling AI 助理。根據使用者提問，輸出一個且僅一個合法 JSON。

可用工具：
- get_stock_price：查詢股票即時價格 → 參數 {"symbol": "AAPL"}
- get_weather：查詢城市即時天氣 → 參數 {"city": "Taipei"}
- get_news：查詢分類新聞 → 參數 {"category": "tech"}（可選 tech / finance / sports）

輸出格式：
情況一：需要工具 → {"tool": "<工具名稱>", "arguments": {<參數鍵值>}}
情況二：不需要工具（閒聊、與工具無關等）→ {"final": "<繁體中文回答>"}

【強制規則】
1. 必須呼叫工具。
2. 不得建議使用者去外部網站查詢。
"""

print(SYSTEM_PROMPT_R1)

## 2. Round 2 System Prompt

Round 2 的目標：把 Python 查到的資料餵給模型，讓它生成自然語言回答。

`build_round2_prompt` 是一個會根據工具結果動態組裝 prompt 的函式。重點：

- 工具回傳 `ok=true` 時，必須使用 `data` 中的具體數值回答。
- 工具回傳 `ok=false` 時，必須回「查無該資料」，不能憑空編造。
- 仍然只輸出 JSON `{"final": "..."}`。

In [ ]:
def build_round2_prompt(function_name: str, result_json: str) -> str:
    return f"""工具「{function_name}」查詢結果如下：
{result_json}

請根據以上資料用繁體中文回答使用者的問題。輸出格式：
{{"final": "<自然語言回答>"}}

【規則】
- 當 ok=true：必須引用 data 中的具體數值或內容（例如：價格、溫度、新聞標題），不得另外編造資訊。
- 當 ok=false：直接輸出 {{"final": "查無該資料。"}}。
- 只輸出 JSON，不要有任何其他文字。
"""

print(build_round2_prompt("get_stock_price", '{"ok": true, "data": {"price": 150.25, "currency": "USD"}}'))

## 3. Tool Schema

Tool schema 是給模型看的「工具說明書」。模型會根據 `description` 和 `parameters` 來判斷：

- 我該用哪個工具？
- 我該抽哪些參數？

好的 description 會具體寫出**使用情境**和**範例參數**，而不是只寫工具名稱。

底下定義 3 個基礎工具，你需要再做 1 個自訂工具（TODO4）。

In [ ]:
# TODO3: 請填寫各個 tool 的 description、parameters 內的 `type` 和 `required`
# Hint: 可參考 `get_stock_avg_price`

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "TODO3",
            "parameters": {
                "type": "object",
                "properties": {
                    "symbol": {
                        "type": "TODO3",
                        "description": "股票代碼，例如 'AAPL'、'TSLA'、'TSMC'，或中文別名如『蘋果』、『特斯拉』、『台積電』",
                    }
                },
                "required": ["TODO3"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "TODO3",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "TODO3",
                        "description": "城市英文名，例如 'Taipei'、'Tokyo'、'London'，或中文別名如『台北』、『東京』、『倫敦』",
                    }
                },
                "required": ["TODO3"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_news",
            "description": "TODO3",
            "parameters": {
                "type": "object",
                "properties": {
                    "category": {
                        "type": "TODO3",
                        "description": "新聞類別：'tech'（科技）、'finance'（財經）、'sports'（運動）",
                        "enum": ["tech", "finance", "sports"],
                    }
                },
                "required": ["TODO3"],
            },
        },
    },
    # TODO4: 實作自訂工具
    # 範例：計算兩支股票的平均股價
    # {
    #     "type": "function",
    #     "function": {
    #         "name": "get_stock_avg_price",
    #         "description": "計算兩支股票的平均股價（兩支股票必須使用相同貨幣）。當使用者要比較或求兩股票均價時使用。",
    #         "parameters": {
    #             "type": "object",
    #             "properties": {
    #                 "symbol_a": {"type": "string", "description": "第一支股票代碼"},
    #                 "symbol_b": {"type": "string", "description": "第二支股票代碼"},
    #             },
    #             "required": ["symbol_a", "symbol_b"],
    #         },
    #     },
    # },
]

print(f"共定義 {len(TOOLS)} 個工具：", [t["function"]["name"] for t in TOOLS])

## 4. handle_tool_call：路由與資料庫查詢

Round 1 的模型輸出告訴我們「要呼叫哪個工具、用什麼參數」，但**真正的查詢動作要在 Python 端做**。`handle_tool_call` 負責這個：

- 根據 `function_name` 決定走哪個分支
- 用 `lookup_db` 從 `mock_data.json` 撈資料
- 統一回傳格式：`{"ok": true, "data": ...}` 或 `{"ok": false, "error": "not_found"}`

**重要**：自訂工具的結果必須**從 mock_data 推導**，不能 hardcode 答案。

In [ ]:
def handle_tool_call(function_name: str, function_args: dict, db: dict) -> str:
    result = None

    if function_name == "get_stock_price":
        symbol = function_args.get("symbol", "")
        result = lookup_db(db.get("stocks", {}), symbol)

    elif function_name == "get_weather":
        city = function_args.get("city", "")
        result = lookup_db(db.get("weather", {}), city)

    elif function_name == "get_news":
        category = function_args.get("category", "")
        result = lookup_db(db.get("news", {}), category)

    # TODO4: 實作自訂工具
    # 範例（已註解，目前不會執行）──
    # 下方為「兩支股票平均股價」的完整實作示範。
    # 請參考此格式，自行實作一個「get_stock_avg_price 以外」的工具。
    # elif function_name == "get_stock_avg_price":
    #     a = lookup_db(db.get("stocks", {}), function_args.get("symbol_a", ""))
    #     b = lookup_db(db.get("stocks", {}), function_args.get("symbol_b", ""))
    #     if a and b and a.get("currency") == b.get("currency"):
    #         result = {
    #             "avg_price": round((a["price"] + b["price"]) / 2, 2),
    #             "currency": a["currency"],
    #             "symbols": [function_args["symbol_a"], function_args["symbol_b"]],
    #         }

    else:
        return json.dumps({"ok": False, "error": "unknown_tool"}, ensure_ascii=False)

    if result is not None:
        return json.dumps({"ok": True, "data": result}, ensure_ascii=False)
    return json.dumps({"ok": False, "error": "not_found"}, ensure_ascii=False)


# 簡單測試一下
print(handle_tool_call("get_stock_price", {"symbol": "AAPL"}, db))
print(handle_tool_call("get_stock_price", {"symbol": "MSFT"}, db))

## 5. run_agent：兩輪 Function Calling 主流程

把上面四件事串起來。流程：

1. **Round 1**：用 `SYSTEM_PROMPT_R1` 問模型，解析 JSON。
   - 如果是 `{"final": ...}` → 直接回答（閒聊路徑）
   - 如果是 `{"tool": ..., "arguments": ...}` → 進入 Round 2
2. **工具呼叫**：用 `handle_tool_call` 查 mock_data。
3. **Round 2**：用 `build_round2_prompt(工具名, 工具結果)` 問模型，要它寫成自然語言。

回傳 `{"model_call": ..., "final_answer": ...}`，這就是寫進 `fc_log.json` 的內容。

In [ ]:
def run_agent(query: str, db: dict) -> dict:
    # ── Round 1：判斷是否使用工具 ──
    r1_messages = [
        {"role": "system", "content": SYSTEM_PROMPT_R1},
        {"role": "user", "content": query},
    ]
    r1_raw = call_llm(r1_messages, max_tokens=1024)
    r1_json = parse_json_output(r1_raw)

    # 閒聊路徑
    if "final" in r1_json and "tool" not in r1_json:
        return {"model_call": "無呼叫工具", "final_answer": r1_json["final"]}

    function_name = r1_json.get("tool", "")
    function_args = r1_json.get("arguments", {})
    if not function_name:
        return {"model_call": "無呼叫工具（格式錯誤）", "final_answer": r1_raw}

    model_call_info = f"{function_name}({json.dumps(function_args, ensure_ascii=False)})"

    # ── 工具呼叫（Python 端查 mock_data）──
    tool_result = handle_tool_call(function_name, function_args, db)

    # ── Round 2：用工具結果生成自然語言 ──
    r2_messages = [
        {"role": "system", "content": build_round2_prompt(function_name, tool_result)},
        {"role": "user", "content": query},
    ]
    r2_raw = call_llm(r2_messages, max_tokens=2048)
    r2_json = parse_json_output(r2_raw)
    final_answer = r2_json.get("final", r2_raw)

    return {"model_call": model_call_info, "final_answer": final_answer}

## 6. 單題除錯

正式跑 5 題前，建議先用單題確認 prompt 與 schema 是否合理。

In [ ]:
result = run_agent("台北的天氣如何？", db)
print("model_call :", result["model_call"])
print("final_answer:", result["final_answer"])

In [ ]:
# TODO4: 完成 TODO4 前兩段後，請寫一個能觸發你的自訂工具的 query 來驗證
# 範例（示範格式，AAPL+TSLA 是針對註解中的 get_stock_avg_price）：
# result = run_agent("請問 AAPL 跟 TSLA 兩支股票的平均股價？", db)
# print("model_call :", result["model_call"])
# print("final_answer:", result["final_answer"])

# 你的測試（請改成觸發你自訂工具的 query）：
# result = run_agent("<你的 query>", db)
# print("model_call :", result["model_call"])
# print("final_answer:", result["final_answer"])

## 7. 正式測驗（5 題，輸出 fc_log.json）

`run_tests` 會跑下列 5 題並把結果寫到 `fc_log.json`，這是你最終要繳交的檔案：

| Q | 題目 | 測什麼 |
|---|---|---|
| Q1 | AAPL 股價 | 工具選擇與參數抽取 |
| Q2 | Taipei 天氣 | 工具選擇與參數抽取 |
| Q3 | tech 新聞 | 工具選擇與參數抽取 |
| Q4 | MSFT 股價（資料庫沒有）| 抗幻覺：能否回「查無資料」 |
| Q5 | 「你是什麼人工智慧模型？」 | 能否判斷不該呼叫工具 |

In [ ]:
results = run_tests(run_agent, db)

In [ ]:
for i, result in enumerate(results):
    print(f"Q{i+1}：{result['query']}")
    print(f"A{i+1}：{result['final_answer']}")
    print("="*30)

## 8. 加分題：跨模型比較（+10）            
跑第二個模型，產出另一份 fc_log，在報告中分析兩個模型的差異（工具選擇、參數抽取、抗幻覺、閒聊判斷）。

In [ ]:
# 加分題：跑第二個模型

model_name = "llama-3.3-70b-versatile" # 你可以自己選別的
set_model(model_name)
run_tests(run_agent, db, output_file=f"fc_log_{model_name}.json")